<a href="https://colab.research.google.com/github/mandakini-15/NLP-Assignment/blob/main/NLP_Assignment_1_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install requests pandas spacy nltk matplotlib wordcloud emoji contractions psaw
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Importing of the required Libraries

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, ne_chunk
from collections import Counter, defaultdict
import pandas as pd
import contractions
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
import emoji
import spacy
from nltk.tokenize import TweetTokenizer, word_tokenize #handles social text tokens like emoticons/lengthening
from nltk.stem import SnowballStemmer #Snowball for English stemming
warnings.filterwarnings('ignore')

# Download required NLTK data
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')
nltk.download('punkt_tab') # Added to fix the LookupError

nlp = spacy.load("en_core_web_sm")
tokenizer = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)   #preserves emoticons, reduces repeated letters, lowercases via preserve_case flag set False
stemmer = SnowballStemmer('english')
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Dataset Details:**

  **Dataset:** Sentiment140 dataset

  **Description:** It containing 1.6 million tweets preprocessed to remove emoticons and ready for sentiment analysis tasks, with labels of 0 (negative) or 4 (positive), ideal for training models to detect polarity in text.
   
  **Source:** Dataset has been downloaded from Kaggle

In [21]:
c = 'training.1600000.processed.noemoticon.csv' #the local path to dataset
cols = ['target','id','date','flag','user','text']
df = pd.read_csv(c, encoding='latin-1', names=cols, on_bad_lines='skip', engine='python') #reading with latin-1 to handle special characters; use column ordering from dataset
df = df.sample(n=20000, random_state=42).reset_index(drop=True)
df['text'] = df['text'].astype(str)

display(df.head(10))

,target,id,date,flag,user,text
0,0,2200003196,Tue Jun 16 18:18:12 PDT 2009,NO_QUERY,LaLaLindsey0609,@chrishasboobs AHHH I HOPE YOUR OK!!!
1,0,1467998485,Mon Apr 06 23:11:14 PDT 2009,NO_QUERY,sexygrneyes,"@misstoriblack cool , i have no tweet apps fo..."
2,0,2300048954,Tue Jun 23 13:40:11 PDT 2009,NO_QUERY,sammydearr,@TiannaChaos i know just family drama. its la...
3,0,1993474027,Mon Jun 01 10:26:07 PDT 2009,NO_QUERY,Lamb_Leanne,School email won't open and I have geography ...
4,0,2256550904,Sat Jun 20 12:56:51 PDT 2009,NO_QUERY,yogicerdito,upper airways problem
5,0,2052380495,Sat Jun 06 00:32:16 PDT 2009,NO_QUERY,Yengching,Going to miss Pastor's sermon on Faith...
6,4,1983449090,Sun May 31 13:10:36 PDT 2009,NO_QUERY,jessig06,on lunch....dj should come eat with me
7,0,2245479748,Fri Jun 19 16:11:29 PDT 2009,NO_QUERY,felicityfuller,@piginthepoke oh why are you feeling like that?
8,0,1770705699,Mon May 11 22:01:32 PDT 2009,NO_QUERY,stephiiheyy,gahh noo!peyton needs to live!this is horrible
9,4,1970386589,Sat May 30 03:39:34 PDT 2009,NO_QUERY,wyndwitch,@mrstessyman thank you glad you like it! There...


**DATASET PREPARATION**
  

In [22]:
print("\n\n" + "="*80)
print("Cleaning")
print("="*80)

# Regex definitions
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r'@\w+') # To remove only @ from teh mention
HASHTAG_RE = re.compile(r"#(\w+)")
NON_ALPHANUM_RE = re.compile(r"[^a-zA-Z0-9\s]")
REPEATED_CHARS_RE = re.compile(r"(.)\1{2,}")
ASCII_EMOTICONS = re.compile(r"[:;=8][\-~]?[)(DPpOo/\\]|[)(DPpOo/\\][\-~]?[:;=8]")
PUNCTUATIONS = re.compile(r"[^\w\s]")

def clean_text(text, remove_emoji=True, remove_numbers=True, keep_hashtag_word=True):
    if not isinstance(text, str):
        return ""

    # Expand contractions
    text = contractions.fix(text)

    # Optionally remove emoji; to keep them use emoji.demojize(text)
    if remove_emoji:
        text = emoji.replace_emoji(text, replace='')
    # Remove URLs
    text = URL_RE.sub(" ", text)

    # Remove extra ASCII emoticons
    text =  ASCII_EMOTICONS.sub('', text)

    # Remove extra Puntuations
    text =  PUNCTUATIONS.sub('', text)

    # Remove mentions (rare in YouTube comments but keep for generality)
    text = MENTION_RE.sub(" ", text)

    # Normalize hashtags - keep the word following '#'
    if keep_hashtag_word:
        text = HASHTAG_RE.sub(r"\1", text)
    else:
        text = re.sub(r"#\w+", " ", text)

    # Remove numbers optionally
    if remove_numbers:
        text = re.sub(r"\d+", " ", text)

    # Remove non-alphanumeric (keep spaces)
    text = NON_ALPHANUM_RE.sub(" ", text)

    # Reduce long repeated chars: sooo -> soo
    text = REPEATED_CHARS_RE.sub(r"\1\1", text)

    # Lowercase
    text = text.lower()

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    #Tokenize
    tokens = word_tokenize(text)

    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    return ' '.join(tokens)

# Apply cleaning
df['clean'] = df['text'].apply(clean_text)
print("Original -> Clean (sample):\n")
for i in range(min(5, len(df))):
    print("ORIG:", df.loc[i,'text'])
    print("CLEAN:", df.loc[i,'clean'])
    print("---")
df.head(5)



Cleaning
Original -> Clean (sample):

ORIG: @chrishasboobs AHHH I HOPE YOUR OK!!! 
CLEAN: chrishasboobs ahh hope
---
ORIG: @misstoriblack cool , i have no tweet apps  for my razr 2
CLEAN: misstoriblack cool tweet apps razr
---
ORIG: @TiannaChaos i know  just family drama. its lame.hey next time u hang out with kim n u guys like have a sleepover or whatever, ill call u
CLEAN: tiannachaos know family drama lamehey next time hang kim guys like sleepover whatever ill call
---
ORIG: School email won't open  and I have geography stuff on there to revise! *Stupid School* :'(
CLEAN: school email open geography stuff revise stupid school
---
ORIG: upper airways problem 
CLEAN: upper airways problem
---


,target,id,date,flag,user,text,clean
0,0,2200003196,Tue Jun 16 18:18:12 PDT 2009,NO_QUERY,LaLaLindsey0609,@chrishasboobs AHHH I HOPE YOUR OK!!!,chrishasboobs ahh hope
1,0,1467998485,Mon Apr 06 23:11:14 PDT 2009,NO_QUERY,sexygrneyes,"@misstoriblack cool , i have no tweet apps fo...",misstoriblack cool tweet apps razr
2,0,2300048954,Tue Jun 23 13:40:11 PDT 2009,NO_QUERY,sammydearr,@TiannaChaos i know just family drama. its la...,tiannachaos know family drama lamehey next tim...
3,0,1993474027,Mon Jun 01 10:26:07 PDT 2009,NO_QUERY,Lamb_Leanne,School email won't open and I have geography ...,school email open geography stuff revise stupi...
4,0,2256550904,Sat Jun 20 12:56:51 PDT 2009,NO_QUERY,yogicerdito,upper airways problem,upper airways problem


In [24]:
print("\n\n" + "="*80)
print("Normalization")
print("="*80)

lemmatizer = WordNetLemmatizer()

def normalize_text_lemm(text):
    """
    Normalize text using lemmatization and handle social media specifics
    """
    tokens = word_tokenize(text)

    # Lemmatization
    lemmatized = [lemmatizer.lemmatize(token, pos='v') for token in tokens]
    lemmatized = [lemmatizer.lemmatize(token, pos='n') for token in lemmatized]

    return ' '.join(lemmatized)

df['normalized_text_lemm'] = df['clean'].map(normalize_text_lemm)
df.head(5)

def normalize_text_stem(text):
    """
    Normalize text using stemming
    """
    return " ".join(stemmer.stem(tok) for tok in tokenizer.tokenize(text))

df['normalized_text_stem'] = df['clean'].map(normalize_text_stem)
df.head(5)



Normalization


,target,id,date,flag,user,text,clean,normalized_text_lemm,normalized_text_stem
0,0,2200003196,Tue Jun 16 18:18:12 PDT 2009,NO_QUERY,LaLaLindsey0609,@chrishasboobs AHHH I HOPE YOUR OK!!!,chrishasboobs ahh hope,chrishasboobs ahh hope,chrishasboob ahh hope
1,0,1467998485,Mon Apr 06 23:11:14 PDT 2009,NO_QUERY,sexygrneyes,"@misstoriblack cool , i have no tweet apps fo...",misstoriblack cool tweet apps razr,misstoriblack cool tweet apps razr,misstoriblack cool tweet app razr
2,0,2300048954,Tue Jun 23 13:40:11 PDT 2009,NO_QUERY,sammydearr,@TiannaChaos i know just family drama. its la...,tiannachaos know family drama lamehey next tim...,tiannachaos know family drama lamehey next tim...,tiannachao know famili drama lamehey next time...
3,0,1993474027,Mon Jun 01 10:26:07 PDT 2009,NO_QUERY,Lamb_Leanne,School email won't open and I have geography ...,school email open geography stuff revise stupi...,school email open geography stuff revise stupi...,school email open geographi stuff revis stupid...
4,0,2256550904,Sat Jun 20 12:56:51 PDT 2009,NO_QUERY,yogicerdito,upper airways problem,upper airways problem,upper airway problem,upper airway problem
